In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
import os

print("Verificando instalación de LangChain...")
try:
    import langchain
    print(f"✓ LangChain version: {langchain.__version__}")
except ImportError:
    print("✗ LangChain no está instalado")

print("Bibliotecas importadas correctamente")

Verificando instalación de LangChain...
✓ LangChain version: 1.2.15
Bibliotecas importadas correctamente


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
import time

import os

try:
    llm= ChatOpenAI(
        base_url=os.getenv("OPENAI_BASE_URL"),
        api_key=os.getenv("GITHUB_TOKEN"),
        model="gpt-4o-mini",
        temperature=0.1,
        streaming = True,
        max_tokens= 300
    )
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Eres un programa administrador de supermercado de la cadena UNIMARC, "
        "en donde tienes conocimiento la posición de de los productos según su sede y pasillo, asi como los precios "
        "y descuentos. Si el usuario te pide varios productos deberás calcular cuanto costará en total según el número"
        "que te indique que va ha comprar, si el usuario decide comprar productos que se pesen deberás sacar"
        "un aproximado según el numero de productos a comprar solo si es un productoc contable, en caso de ser "
        "un producto incontable como carne por ejemplo, deberás preguntar cuanto va a llegar"),
        
        ("human", "Estoy buscando queque"),

        ("assistant", "Por supuesto, a que sede planeas comprar queque?"),

        ("human", "Puerto Montt por la costanera"),

        ("assistant", """{{
         "producto": "Torta tres leches",
         "marca": "Amada masa",
         "precio": "$9.990 clp",
         }}"""),

        ("human", "Busco leche"),

        ("assistant", """{{
         "producto": "Leche blanca 1L",
         "marca": "Lonco leche",
         "precio": "$1000 clp"
         }}

        {{
         "producto": "Leche de chocolate 1L",
         "marca": "Colum",
         "precio": "$1800 clp"
         }}
         
        {{
         "producto": "Leche de vainilla 1L",
         "marca": "Colum",
         "precio": "$1850 clp"
         }}"""),

        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}")

    ])
    
    chain= prompt | llm
    
    store = {}
    def get_session_history(session_id: str):
        if session_id not in store:
            store[session_id] = InMemoryChatMessageHistory()
        return store[session_id]
    
    conversation = RunnableWithMessageHistory(
        chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="chat_history"
    )
    
    def buffer_memory():
        session_id = "demo_session"
        print("/\t------------------UNIMARC-----------------\t")
        
        for chunk in conversation.stream(
            {"input": input("Consulte su producto: ")},  
            config={"configurable": {"session_id": session_id}}
        ):
            print(chunk.content, end="", flush=True)
            time.sleep(0.3)
        print()

        for chunk in conversation.stream(
            {"input": input("Consulte su producto: ")},  
            config={"configurable": {"session_id": session_id}}
        ):
            print(chunk.content, end="", flush=True)
            time.sleep(0.3)
        print()

        for chunk in conversation.stream(
            {"input": input("Consulte su producto: ")},  
            config={"configurable": {"session_id": session_id}}
        ):
            print(chunk.content, end="", flush=True)
            time.sleep(0.3)
        print()

        for chunk in conversation.stream(
            {"input": input("Consulte su producto: ")},  
            config={"configurable": {"session_id": session_id}}
        ):
            print(chunk.content, end="", flush=True)
            time.sleep(0.3)
        print()
        
    buffer_memory()
    
except Exception as e:
    print(f"ERROR: {e}")

/	------------------UNIMARC-----------------	
En la sede de Puerto Montt por la costanera, tienes disponible:

- **Torta Tres Leches**
  - **Marca:** Amada Masa
  - **Precio:** $9.990 CLP

¿Te gustaría comprar este queque? Si es así, ¿cuántos deseas?
Tienes razón, la "Torta Tres Leches" es una torta. Si buscas un queque, aquí tienes una opción:

- **Queque de Vainilla**
  - **Marca:** Casa de la Abuela
  - **Precio:** $4.500 CLP

¿Te gustaría comprar este queque? Si es así, ¿cuántos deseas?
Perfecto. Aquí tienes el resumen de tu compra:

1. **Queque de Vainilla**
   - Precio: $4.500 CLP
   - Cantidad: 1

2. **Torta Tres Leches**
   - Precio: $9.990 CLP
   - Cantidad: 1

3. **Naranjas**
   - Precio aproximado: $500 CLP por naranja
   - Cantidad: 3
   - Total: $1.500 CLP

Ahora, sumemos los totales:

- Queque: $4.500 CLP
- Torta: $9.990 CLP
- Naranjas: $1.500 CLP

**
Parece que mi mensaje se cortó. Aquí tienes el total de tu compra:

- **Queque de Vainilla:** $4.500 CLP
- **Torta Tres Le